# 02 · Forecast backtest

Delivery day D is forecast at 12:00 on D-1, monthly refit on an expanding window.
Model selection on the validation year (2024-11 – 2025-10), final evaluation once on the test year (2025-11 – 2026-10).

In [1]:
from datetime import date

import polars as pl
import plotly.io as pio

from energy_forecast import features, forecast, plots
from energy_forecast.data import ROOT, load
from energy_forecast.features import BASE_FEATURES
from energy_forecast.forecast import PARAMS, TEST, VAL, Config

pio.templates.default = "plotly_white"
df = features.build(load())

## Model selection on the validation year

In [2]:
CONFIGS = [
    Config("base", features=tuple(BASE_FEATURES), anchor=None, window_days=None),
    Config("anchor", features=tuple(BASE_FEATURES), window_days=None),
    Config("anchor_feat", window_days=None),
    Config("anchor_feat_3y", window_days=3 * 365),
    Config("anchor_feat_2y", window_days=2 * 365),
    Config("anchor_feat_l2", window_days=None, params={**PARAMS, "objective": "l2"}),
]

In [ ]:
%%time
val = pl.concat(
    [forecast.seasonal_naive(df, 24, VAL), *(forecast.backtest(df, cfg, VAL) for cfg in CONFIGS)]
)
forecast.score(df, val)

In [ ]:
plots.mae_by(df, val, "month")

## Test year

`forecast.DEFAULT` holds the selected configuration.

In [ ]:
%%time
test = pl.concat(
    [
        forecast.seasonal_naive(df, 24, TEST),
        forecast.seasonal_naive(df, 168, TEST),
        forecast.backtest(df, forecast.DEFAULT, TEST),
    ]
)
forecast.score(df, test)

In [ ]:
plots.mae_by(df, test, "month")

In [ ]:
plots.mae_by(df, test, "hour")

In [ ]:
plots.forecast_lines(df, test, date(2026, 6, 15), date(2026, 6, 21))

In [ ]:
plots.forecast_lines(df, test, date(2026, 10, 12), date(2026, 10, 18))

## Chronos-2 (zero-shot)

Context: last 2048 hours up to the end of D-1. With covariates: TSO forecasts for load, wind, solar and residual load, past and future.
No tuning, so evaluating directly on the test year is admissible. The ensemble is a fixed 50/50 average, chosen a priori.

In [ ]:
from energy_forecast import foundation

pipeline = foundation.load_pipeline()

In [ ]:
%%time
chronos = pl.concat(
    [
        foundation.backtest(pipeline, df, TEST, covariates=False),
        foundation.backtest(pipeline, df, TEST, covariates=True),
    ]
)
test = pl.concat([test, chronos])
test = pl.concat([test, forecast.ensemble(test, ["lightgbm", "chronos2_cov"])])
forecast.score(df, test)

In [ ]:
plots.mae_by(df, test, "month")

In [ ]:
plots.forecast_lines(df, test.filter(pl.col("model").is_in(["lightgbm", "chronos2_cov", "ensemble"])), date(2026, 10, 12), date(2026, 10, 18))

## Save predictions

In [ ]:
out = ROOT / "data/processed"
out.mkdir(parents=True, exist_ok=True)
test.write_parquet(out / "backtest.parquet")